# Crear y Compilar la Red 1D-CNN + LSTM para MyoTensor (ESP32-S3)

Este cuaderno define la arquitectura temporal **1D-CNN + LSTM** para clasificación sEMG a 1 solo canal.
- **Entrada:** Ventana de 300 ms ($F_s = 1000$ Hz, tensor $(300, 1)$).
- **Extracción de características:** 2 Bloques `Conv1D + BatchNorm + MaxPool + SpatialDropout`.
- **Dinámica temporal:** Capa `LSTM(64)` unidireccional con `unroll=True` para compatibilidad con **TensorFlow Lite Micro (INT8)** en ESP32-S3.
- **Salida:** Clasificador `Dense(32) -> Dense(4, softmax)` (Reposo, Palma Abierta, Puño, Paz).


In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
from pathlib import Path

# Carga de variables de entorno desde .env
def load_env_variables():
    start_dir = Path(os.getcwd())
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()
models_dir = os.environ["MODELS_DL_PROTO"]
os.makedirs(models_dir, exist_ok=True)


I0000 00:00:1787692290.890511   71011 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787692291.268331   71011 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787692292.645897   71011 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ Archivo .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


In [2]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D, 
    LSTM, Dropout, Dense, SpatialDropout1D
)

def build_cnn_lstm_model(
    window_size=300, 
    num_channels=1, 
    num_classes=4, 
    conv_filters=(32, 64), 
    kernel_size=3, 
    lstm_units=64, 
    dropout_rate=0.3, 
    learning_rate=1e-3
):
    """
    Construye y compila el modelo 1D-CNN + LSTM para clasificación sEMG.
    Diseñado para ser compatible con TensorFlow Lite Micro (INT8) en ESP32-S3.
    """
    inputs = Input(shape=(window_size, num_channels), name="input_semg")
    
    x = inputs
    # Bloques Convolucionales (Extracción de características locales)
    for i, filters in enumerate(conv_filters):
        x = Conv1D(
            filters=filters, 
            kernel_size=kernel_size, 
            padding="same", 
            activation="relu",
            name=f"conv1d_{i+1}"
        )(x)
        x = BatchNormalization(name=f"bn_{i+1}")(x)
        x = MaxPooling1D(pool_size=2, name=f"pool_{i+1}")(x)
        x = SpatialDropout1D(dropout_rate, name=f"spatial_dropout_{i+1}")(x)
    
    # Capa Recurrente (Dinámica temporal unidireccional desenrollada para TFLite Micro)
    x = LSTM(lstm_units, return_sequences=False, unroll=True, name="lstm_layer")(x)
    x = Dropout(dropout_rate, name="dropout_dense")(x)
    
    # Capas de Clasificación
    x = Dense(32, activation="relu", name="dense_features")(x)
    outputs = Dense(num_classes, activation="softmax", name="output_gestures")(x)
    
    model = Model(inputs=inputs, outputs=outputs, name="CNN_LSTM_MyoTensor")
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


In [3]:
# Instanciar y revisar resumen de la arquitectura
model = build_cnn_lstm_model(window_size=300, num_channels=1, num_classes=4)
model.summary()

# Guardar modelo base en formato .keras
model_save_path = os.path.join(models_dir, "myotensor_proto_net_lstm.keras")
print(f"Guardando modelo inicial en: {model_save_path} ...")
model.save(model_save_path)
print("¡Modelo CNN-LSTM base guardado con éxito!")


W0000 00:00:1787692293.418258   71011 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "CNN_LSTM_MyoTensor"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_semg (InputLayer)     [(None, 300, 1)]          0         
                                                                 
 conv1d_1 (Conv1D)           (None, 300, 32)           128       
                                                                 
 bn_1 (BatchNormalization)   (None, 300, 32)           128       
                                                                 
 pool_1 (MaxPooling1D)       (None, 150, 32)           0         
                                                                 
 spatial_dropout_1 (Spatial  (None, 150, 32)           0         
 Dropout1D)                                                      
                                                                 
 conv1d_2 (Conv1D)           (None, 150, 64)           6208      
                                                